# Regenerar tablas y figuras de A sin ejecutar HiGHS

Este notebook usa una corrida ya terminada. No genera escenarios ni resuelve
ningún MILP. Puede usar la carpeta local del experimento completo o descargar
sólo `metrics.csv` y `manifest.json` de la revisión publicada por el autor.
Los datos originales se conservan. Las figuras se guardan en PDF/SVG vectorial
y PNG de 1200 dpi. Los puntos y las estadísticas no se suavizan ni interpolan.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, subprocess, sys
from urllib.request import urlopen

RUNTIME_REVISION = '303338dfb98667a254f272129c79b0f72f11fcaa'
EXPECTED_SHA256 = {
    "frozen_generator_A.py": "d1eada7a7557203598f165be50b60faa4e9ef5e572bd64327d430848559841a6",
    "frozen_generator_B.py": "c33fbb583bceb3582f968f219abc8f5d1b4c5aaf17fa5ac8801cf66c32570297",
    "frozen_ttap_core.py": "d2d022eaf39cf91ab0d68eba3d95c131a2f90b6408d8e32472e29dc331b114a5",
    "ttap_warm_start.py": "96a9dc50863b7bb41ef34359cf4cf6cd143880cfd14a9970044396620d0405f6",
    "objective_profiles.py": "a655b2368b94b96d366fb3919e4300e0d8715efb5c2c88b2b00bbb8564b0eafe",
    "study_runner.py": "e02a59f2819730de6e728edbf13e02a630839ee6113e57f46060989ae6bf15bd",
    "study_analysis.py": "a673fdece4141fec6de81f98c3e955d962e6d415b71097bd8aec6d57ab3bafc4",
    "figure5_and_analysis.py": "8b611fa972678f051710c89484935a3eb84d26409c694c67e6bddd16f5757321",
    "public_sources.py": "4e1a545646f1ba46bf5a78f0930f39fac318f7ac304fe3874f41c2305ac36f1e",
    "talcahuano_three_methods.py": "ecabb5c3e4ba6af98a35dc4845a840891e0f6b7dacb8bfa0bfdbd44fe1c69d9d",
    "ablation_recovery.py": "e41a7a35efce97ada50c65493d7cafeabc47564e5263a9d8de3de7918331fc97",
    "requirements_execution.txt": "37b0105f36958ae5a61e8b2628f96339e1c9f9eff1306eee3748357d63766876",
    "paper_experiment_a.py": "cccdb79f0db87b3ab78398a5663dbd256e1b3e7a69cf63b3e4d91614fda05b9c",
    "paper_a_report.py": "b31b113b554a56e605d9f83330f7d22daaf330883ef2fa7886834193c3bbc403",
    "paper_experiment_b.py": "9c44ac82fc0becb861e3667c47b6340c27bf87665844489e24cae70778804d68",
    "paper_b_report.py": "d796e72bcf97a0783db1ebc56f2831b7ce9739cb8316525946ef21acb083ff17"
}

def _ttap_canonical_bytes(content):
    # Git may check out text with CRLF on Windows. No other text is discarded.
    return content.replace(b'\r\n', b'\n')


def _ttap_prepare_runtime(revision, expected_hashes, *, root=None, opener=None):
    import tempfile

    root = Path.cwd() if root is None else Path(root)
    opener = urlopen if opener is None else opener
    signature = hashlib.sha256(
        json.dumps(expected_hashes, sort_keys=True).encode('utf-8')
    ).hexdigest()[:12]
    runtime = root / ('ttap_runtime_' + revision[:10] + '_' + signature)
    runtime.mkdir(parents=True, exist_ok=True)
    candidates = [root / 'TTAP_WarmStart', root / 'Resuelto_Paper' / 'TTAP_WarmStart',
                  root, root / ('ttap_runtime_' + revision[:10])]
    base = ('https://raw.githubusercontent.com/educarrascov/DISC_TESIS/'
            + revision + '/Resuelto_Paper/TTAP_WarmStart/')
    counts = {'cache': 0, 'local': 0, 'download': 0}

    def matching_bytes(path, expected):
        if not path.is_file():
            return None
        content = _ttap_canonical_bytes(path.read_bytes())
        return content if hashlib.sha256(content).hexdigest() == expected else None

    for name, expected in expected_hashes.items():
        path = runtime / name
        content = matching_bytes(path, expected)
        if content is not None:
            counts['cache'] += 1
            # Files written here always use LF, including a cache copied from Windows.
            if path.read_bytes() == content:
                continue
        else:
            for candidate in candidates:
                content = matching_bytes(candidate / name, expected)
                if content is not None:
                    counts['local'] += 1
                    break
            if content is None:
                try:
                    with opener(base + name, timeout=90) as response:
                        content = _ttap_canonical_bytes(response.read())
                except OSError as exc:
                    raise RuntimeError(
                        'No se pudo descargar ' + name + ' de la revisión ' + revision
                        + '. Comprueba la conexión y vuelve a ejecutar esta celda. '
                        + 'Los archivos de tu proyecto no se han modificado.'
                    ) from exc
                if hashlib.sha256(content).hexdigest() != expected:
                    raise RuntimeError('La descarga no coincide con la revisión fijada: '
                                       + name + '. No se ejecutará ese archivo.')
                counts['download'] += 1

        temporary = None
        try:
            with tempfile.NamedTemporaryFile(dir=runtime, prefix=name + '.',
                                             suffix='.tmp', delete=False) as handle:
                temporary = Path(handle.name)
                handle.write(content)
            temporary.replace(path)
        finally:
            if temporary is not None and temporary.exists():
                temporary.unlink()

    print('Fuentes verificadas:', counts)
    return runtime.resolve()


RUNTIME = _ttap_prepare_runtime(RUNTIME_REVISION, EXPECTED_SHA256)
for name in EXPECTED_SHA256:
    if not name.endswith('.py'):
        continue
    loaded = sys.modules.get(name[:-3])
    if loaded is not None:
        location = getattr(loaded, '__file__', None)
        if location is None or Path(location).resolve() != RUNTIME / name:
            raise RuntimeError(
                'Hay módulos TTAP de otra carpeta cargados en memoria. '
                'Reinicia el kernel (Kernel > Restart) y ejecuta desde la primera celda.'
            )


missing_science = [m for m in ['numpy', 'scipy', 'pandas', 'matplotlib']
                   if importlib.util.find_spec(m) is None]
if missing_science:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(RUNTIME / 'requirements_execution.txt')], check=True)
elif importlib.util.find_spec('highspy') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'highspy==1.11.0'], check=True)
sys.path.insert(0, str(RUNTIME))
import highspy
if highspy.Highs().version() != '1.11.0':
    raise RuntimeError('Use highspy==1.11.0 and restart the kernel before the study.')
from IPython.display import display, Image, SVG
from objective_profiles import ObjectiveSpec, professor_examples
from paper_experiment_a import paper_configuration, run_paper_experiment_a
from paper_a_report import export_paper_a
print('Verified runtime:', RUNTIME_REVISION)
print('Local modules:', RUNTIME)
print('Experiment A paper protocol: 1 / 5 / 15 / 60 / 600 s for all three starts')


## Seleccionar los resultados

Si la carpeta local existe, se usa directamente. Si falta y se permite la
descarga, se crea una copia para análisis en `ttap_replot`. No usar esa copia
como checkpoint para reanudar el solver en otro entorno.


In [ ]:
RUN_DIR = Path.cwd() / 'ttap_runs' / 'A_paper_full_9331b25d5427'
DOWNLOAD_IF_MISSING = True
PNG_DPI = 1200
DATA_REVISION = '0be00802518836ed652035dad6d504cfbb614c8a'
DATA_FOLDER = 'Resuelto_Paper/ttap_runs/A_paper_full_9331b25d5427/'
DATA_BLOB_SHA1 = {
    'manifest.json': '59f3b50099eec306b3b1d9510ef60053e3df7086',
    'metrics.csv': 'ef1aadd7a01d7e4841b3cf053803d47da781247c',
}
if not all((RUN_DIR/name).is_file() for name in DATA_BLOB_SHA1):
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError('Selecciona la carpeta que contiene metrics.csv y manifest.json.')
    RUN_DIR = Path.cwd() / 'ttap_replot' / ('A_' + DATA_REVISION[:10])
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    for name, expected in DATA_BLOB_SHA1.items():
        target = RUN_DIR / name
        content = target.read_bytes() if target.is_file() else b''
        actual = hashlib.sha1(b'blob ' + str(len(content)).encode() + b'\0' + content).hexdigest()
        if actual != expected:
            url = 'https://raw.githubusercontent.com/educarrascov/DISC_TESIS/' + DATA_REVISION + '/' + DATA_FOLDER + name
            with urlopen(url, timeout=90) as response:
                content = response.read()
            actual = hashlib.sha1(b'blob ' + str(len(content)).encode() + b'\0' + content).hexdigest()
            if actual != expected:
                raise RuntimeError('El archivo descargado no coincide con la revisión fijada: ' + name)
            temporary = target.with_suffix(target.suffix + '.part')
            temporary.write_bytes(content)
            temporary.replace(target)
print('Results:', RUN_DIR)


## Exportar

A1/A8 y las tablas 1/2 pertenecen a la sección 4.1. A3/A9--A13 permiten
elegir cómo presentar los presupuestos en 4.3. A11 necesita acompañarse de
factibilidad; sus medias excluyen incumbents ausentes. A12 compara la
inyección con cold conservando el mismo greedy como respaldo externo.


In [ ]:
report = export_paper_a(RUN_DIR, png_dpi=PNG_DPI)
PAPER_DIR = report['directory']
display(report['quality'])
print(report['metadata'])
for name in ['A1_solution_quality_coverage_aware', 'A8_computational_effort',
             'A11_budget_optimality_shortfall', 'A12_warm_start_attribution',
             'A13_budget_quality_overview']:
    display(SVG(filename=str(PAPER_DIR / (name + '.svg'))))
print('Overleaf:', PAPER_DIR / 'Section_4_1.tex')
